# CF Modeling — Visualization

Renders flatmap visualizations of CF model outputs using Hedger's vicsompy plotting code.

## Key design choices

**Fig 3a (2D colormap) uses raw split R².** Hedger's Fig 3a uses raw (un-null-corrected)
split R² scores. Raw R² has frac>0 ≈ 82–87%, giving the full smooth coverage seen in
Hedger's figure.

**1D topographic maps use null-corrected R².** `R2_nc = R2_split − R2_null` removes the
global-signal component and reveals the genuine topographic gradient.

**Integration score uses raw R².** The product map √(R²_A × R²_B) uses raw split scores.

**2D bimodal map** is generated here via pycortex (`uber_plot`), not saved as a CIFTI.

## Structure

- **Cell 3**: Global config (mode, paths, template CIFTI)
- **Cell 4**: All helper / plotting functions (`plot_all_for_pair`)
- **Cell 5+**: One cell per ROI pair — calls `plot_all_for_pair(roi_a, roi_b)`

## Pipeline prerequisites

```bash
bash cf_modeling/analysis.sh geometry
bash cf_modeling/analysis.sh avg
python cf_modeling/integration_maps.py --roi-a 3b --roi-b V1 ...
```

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cortex

from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from paths import DATA, OUTPUTS, ROOT
VICSOMPY_REPO = os.environ.get("VICSOMPY_REPO", str(ROOT / "Vicarious_somatotopy"))
if VICSOMPY_REPO not in sys.path:
    sys.path.insert(0, VICSOMPY_REPO)

from vicsompy.vis import Plot, basic_plot
from vicsompy.utils import load_plot_pkg_yaml

# Must point to hedger filestore BEFORE any cortex.db access
PYCORTEX_STORE = str(DATA / "hedger2026")
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for _mod_name, _mod in sys.modules.items():
    if _mod_name.startswith("cortex") and hasattr(_mod, "db"):
        _mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)
CX_SUB = list(cortex.db.subjects.keys())[0]
print(f"Pycortex subject: {CX_SUB}")

In [ ]:
# =============================================================================
# Global config — edit here to change mode or data paths
# =============================================================================
MODE     = "group_average"   # "group_average" | "per_subject"
OUT_BASE = str(OUTPUTS / "cf_modeling")

# Template CIFTI — Hedger pseudo-subject 999999 (default, matches Hedger et al. 2025)
# To switch to our 175-subject average, change to:
#   .../average_sub/sg_psc/group_average_sg_psc_cortex_59k.dtseries.nii
TEMPLATE_CIFTI = str(
    DATA / "preprocessed/average_sub/hedger_sg_psc/group_average_hedger_sg_psc_cortex_59k.dtseries.nii"
)

bm_axis   = nib.load(TEMPLATE_CIFTI).header.get_axis(1)
plotviews = load_plot_pkg_yaml()['plotviews']

print(f"Mode          : {MODE}")
print(f"Pycortex sub  : {CX_SUB}")
print(f"Template      : .../{TEMPLATE_CIFTI.split('/')[-1]}")

In [ ]:
# =============================================================================
# Helper functions — run once, then call plot_all_for_pair() per ROI pair
# =============================================================================

# ── Core utilities ────────────────────────────────────────────────────────────

def to_sphere(arr):
    """Map 108441-grayordinate array → 118584 sphere vertices (zeros at medial wall)."""
    sphere = np.zeros(118584, dtype=np.float32)
    pos = 0
    for name, _, model in bm_axis.iter_structures():
        n = len(model.vertex)
        if "CORTEX_LEFT" in name:
            sphere[model.vertex] = arr[pos:pos+n]
        elif "CORTEX_RIGHT" in name:
            sphere[59292 + model.vertex] = arr[pos:pos+n]
        else:
            pos += n; continue
        pos += n
    return sphere


def load_pair(roi_a, roi_b):
    """Load all R² maps for one ROI pair; return dict with arrays + paths."""
    roi_root = f"{OUT_BASE}/{MODE}/{roi_a}_{roi_b}"
    prep_dir = f"{roi_root}/prep" if MODE == "group_average" else f"{roi_root}/group"
    suffix   = "" if MODE == "group_average" else "_avg"

    def _load(name):
        path = os.path.join(prep_dir, f"{name}{suffix}.npy")
        arr  = np.load(path)
        print(f"  {(name+suffix):<30s}  mean={arr.mean():+.4f}  frac>0={np.mean(arr>0):.1%}")
        return arr

    print(f"\n── Loading {roi_a} × {roi_b} ({MODE}) ──")
    d = dict(roi_root=roi_root, prep_dir=prep_dir)
    d['R2_a']      = _load(f"R2_{roi_a}")
    d['R2_b']      = _load(f"R2_{roi_b}")
    d['R2_a_nc']   = _load(f"R2_{roi_a}_nc")
    d['R2_b_nc']   = _load(f"R2_{roi_b}_nc")
    d['R2_full']   = _load("R2_full")
    d['product']   = _load("product_map")
    _nc = os.path.join(prep_dir, f"product_map_nc{suffix}.npy")
    if os.path.exists(_nc):
        d['product_nc'] = _load("product_map_nc")
    else:
        d['product_nc'] = np.sqrt(
            np.clip(d['R2_a_nc'], 0, None) * np.clip(d['R2_b_nc'], 0, None)
        ).astype(np.float32)
        print(f"  {'product_map_nc':<30s}  (computed inline)")
    return d


# ── Plot functions ────────────────────────────────────────────────────────────

_CMAP_PNG = os.path.join(sys.prefix, "share/pycortex/colormaps/PU_RdBu_covar_alpha.png")
_SURF_DIR = f"{PYCORTEX_STORE}/{CX_SUB}/surfaces"


def plot_fig3a(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir):
    """2D flatmap (Hedger Fig 3a). dat=ROI_B (blue), dat2=ROI_A (red).
    Uses Hedger's exact variance_decomp_full config; only override: subject + no ROI lines."""
    cfg = plotviews['variance_decomp_full'].copy()
    cfg['subject'] = CX_SUB   # required; not in YAML
    cfg.pop('ROIS', None)      # drop ROI outlines
    fig = Plot(**cfg).uber_plot(dat=sph_b_pos, dat2=sph_a_pos)
    out = f"{fig_dir}/fig3a_{roi_a}_{roi_b}_2d_colormap.png"
    fig.savefig(out, bbox_inches='tight', dpi=cfg.get('dpi', 300))
    print(f"Saved: {out}"); plt.show()


def plot_inflated(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir):
    """Inflated surface views (L/R lateral + medial) — same 2D colormap as Fig 3a.

    Orthographic projection onto the Y-Z plane with painter's-algorithm depth
    sorting. Avoids matplotlib 3D distortion entirely.

    HCP inflated coordinate system (verified from GIFTI):
      LH: X in [-67, +1]  (lateral surface faces -X)
      RH: X in [ -2, +67] (lateral surface faces +X)
      Y = anterior-posterior (+Y = anterior)
      Z = inferior-superior (+Z = superior)

    depth_sign +1: camera from -X (L lateral, R medial), farther = larger X
    depth_sign -1: camera from +X (L medial, R lateral), farther = smaller X
    flip_u: negate Y so anterior appears on the anatomically conventional side
    """
    import matplotlib.image as mpimg
    from matplotlib.collections import PolyCollection

    tex = mpimg.imread(_CMAP_PNG)           # (256, 256, 4), float32 in [0, 1]
    H, W = tex.shape[:2]
    vmax = 0.4
    xi = np.clip((np.nan_to_num(sph_b_pos) / vmax * (W - 1)).astype(int), 0, W - 1)
    yi = np.clip((np.nan_to_num(sph_a_pos) / vmax * (H - 1)).astype(int), 0, H - 1)
    yi = (H - 1) - yi
    rgba_all = tex[yi, xi]   # (118584, 4)

    # (title, hemi_file, vertex_offset, depth_sign, flip_u)
    panels = [
        ("L lateral", "inflated_lh", 0,     +1, False),
        ("L medial",  "inflated_lh", 0,     -1, True),
        ("R lateral", "inflated_rh", 59292, -1, True),
        ("R medial",  "inflated_rh", 59292, +1, False),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(14, 4), dpi=150)
    for ax, (label, fname, offset, depth_sign, flip_u) in zip(axes, panels):
        surf   = nib.load(f"{_SURF_DIR}/{fname}.gii")
        coords = surf.darrays[0].data.astype(np.float32)   # (59292, 3)
        faces  = surf.darrays[1].data.astype(np.int32)     # (n_faces, 3)
        rgba_h = rgba_all[offset:offset + 59292]

        # Painter's algorithm: sort faces by depth, paint farthest first
        centroid_x = coords[faces].mean(axis=1)[:, 0]
        order = np.argsort(-centroid_x * depth_sign)

        # Orthographic projection onto Y-Z plane
        u = -coords[:, 1] if flip_u else coords[:, 1]  # horizontal (Y or -Y)
        v = coords[:, 2]                                  # vertical   (Z, superior=up)

        verts   = np.stack([u[faces[order]], v[faces[order]]], axis=-1)  # (nF, 3, 2)
        fcolors = rgba_h[faces[order]].mean(axis=1)                       # (nF, 4)

        coll = PolyCollection(
            verts, facecolors=fcolors,
            edgecolors='none', linewidths=0, rasterized=True,
        )
        ax.add_collection(coll)
        ax.set_xlim(u.min(), u.max())
        ax.set_ylim(v.min(), v.max())
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(label, fontsize=9, pad=2)

    fig.suptitle(f"{roi_a} (red) × {roi_b} (blue)  [inflated]", fontsize=10)
    plt.tight_layout()
    out = f"{fig_dir}/inflated_{roi_a}_{roi_b}_2d_colormap.png"
    fig.savefig(out, bbox_inches='tight', dpi=150)
    print(f"Saved: {out}"); plt.show()


def print_stats_table(roi_a, roi_b, d):
    print(f"\n{'Map':<22}  {'Mean':>8}  {'Frac>0':>7}  {'Max':>8}  {'Min':>8}")
    print("-" * 60)
    for name, arr in [
        (f"R2_{roi_a}",    d['R2_a']),
        (f"R2_{roi_b}",    d['R2_b']),
        (f"R2_{roi_a}_nc", d['R2_a_nc']),
        (f"R2_{roi_b}_nc", d['R2_b_nc']),
        ("R2_full",        d['R2_full']),
        ("product_map",    d['product']),
        ("product_map_nc", d['product_nc']),
    ]:
        print(f"{name:<22}  {arr.mean():>8.4f}  {np.mean(arr>0):>7.1%}"
              f"  {arr.max():>8.4f}  {arr.min():>8.4f}")


# ── Master function ───────────────────────────────────────────────────────────

def plot_all_for_pair(roi_a, roi_b):
    """Load R² maps and generate flatmap + inflated figures for a ROI pair."""
    d = load_pair(roi_a, roi_b)
    fig_dir = f"{d['roi_root']}/figures"
    os.makedirs(fig_dir, exist_ok=True)

    sph_a_pos = np.clip(to_sphere(d['R2_a']), 0, None)
    sph_b_pos = np.clip(to_sphere(d['R2_b']), 0, None)

    print_stats_table(roi_a, roi_b, d)
    plot_fig3a(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir)
    plot_inflated(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir)


print("Helpers defined.")

In [ ]:
# =============================================================================
# 3b × V1  — somatosensory (seed A, red) × visual (seed B, blue)
# =============================================================================
plot_all_for_pair("3b", "V1")

In [ ]:
# =============================================================================
# A1 × V1  — primary auditory (seed A, red) × visual (seed B, blue)
# =============================================================================
plot_all_for_pair("A1", "V1")

In [ ]:
# =============================================================================
# A5 × FFC  — belt auditory (seed A, red) × fusiform face complex (seed B, blue)
# =============================================================================
plot_all_for_pair("A5", "FFC")